In [ ]:
#Install requirements
%pip install -r "../requirements.txt"

In [ ]:
#Import required libraries
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
import scipy
import sys
import glob

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:
#Add the src to the path
sys.path.append(os.path.abspath(os.path.join('..')))

In [ ]:
#Path for the Data
if not os.path.exists("../data/students_dataset.csv"):
    print("Data Path does not exist. Expected in `../data/students_dataset.csv`")
    exit

data = pd.read_csv(os.path.join("../data/students_dataset.csv"))

In [ ]:
#Import the nested cross-validation class
from src.nested_cross_validation import NestedCrossValidation

In [ ]:
#List the estimators with defaulf hyperparametes, set random_state equal to 42
estimators = [('Logistic_Regression', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5, max_iter=5000, random_state=42)),
              ('Gaussian_Naive_Bayes', GaussianNB()),
              ('LDA', LinearDiscriminantAnalysis()),
              ('Random_Forest', RandomForestClassifier(random_state=42)),
              ('LightGMB', LGBMClassifier(random_state=42, verbose=0)), #no logs printed
              ('XGBoost', XGBClassifier(random_state=42)),
              ('CatBoost', CatBoostClassifier(random_state=42, verbose=0)) #no logs printed
]

In [ ]:
#Baseline comparison across algorithms
#Disable the inner loop - no hyperparameter tuning
NCV = NestedCrossValidation(data, estimators=estimators, parameter_space=None, R=10, n_outer=5, seed=42, optimize=False)
NCV.runcv()

In [ ]:
from src.nested_cross_validation import get_models_statistics
from src.nested_cross_validation import plot_metrics

In [ ]:
#Get the statistics for each model and plot the selected metrics
baseline_stats = get_models_statistics(NCV.results, output_filename = "../data/Task3/baseline_models_report.csv")
plot_metrics(baseline_stats, filename = "../figures/Task3/baseline_comparison.png")

In [ ]:
#List the estimators' hyperparameter space
parameter_space = {
    'Logistic_Regression' : lambda trial:{
        'C': trial.suggest_float('C', 0.0001, 100, log=True),
        'l1_ratio' : trial.suggest_float('l1_ratio', 0, 1)
    },
    'Gaussian_Naive_Bayes': lambda trial:{
        'var_smoothing' : trial.suggest_float('var_smoothing', 1e-11, 1e-7, log=True)
    },
    'LDA' : lambda trial :{
        'solver' : trial.suggest_categorical('solver', ['lsqr', 'eigen']),
        'shrinkage' : trial.suggest_float('shrinkage', 0.0, 1.0)
    },
    'Random_Forest': lambda trial : {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth' : trial.suggest_int('max_depth', 3, 12)
    },
    'LightGBM' : lambda trial : {
        'max_depth' : trial.suggest_int('max_depth', 3 , 9),
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves' : trial.suggest_int('num_leaves', 20, 150)
    },
    'XGBoost' : lambda trial : {
        'max_depth' : trial.suggest_int('max_depth', 3 , 9),
        'subsample' : trial.suggest_float('subsample', 0.5, 1.0),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'scale_pos_weight' : trial.suggest_float('scale_pos_weight', 1.0, 3.0)
    },
    'CatBoost' : lambda trial : {
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg' : trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
        'depth' : trial.suggest_int('depth', 4, 10)
    }
}

estimators_for_tuning = [('Logistic_Regression', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=42)),
              ('Gaussian_Naive_Bayes', GaussianNB()),
              ('LDA', LinearDiscriminantAnalysis()),
              ('Random_Forest', RandomForestClassifier(random_state=42)),
              ('LightGBM', LGBMClassifier(random_state=42, verbose=0)), #no logs printed
              ('XGBoost', XGBClassifier(random_state=42, verbosity=0)), #no logs printed
              ('CatBoost', CatBoostClassifier(random_state=42, verbose=0)) #no logs printed 
]


In [ ]:
#Estimate the generalization performance
#Activate the inner loop - optuna hyperparameter tuning
#I will run the class for each estimator seperately because all in one run leads to my computer crash

for name, model in estimators_for_tuning:
    print(f"Starting individual run for: {name}")
    
    #Run the class 
    single_run = NestedCrossValidation(
        data=data, 
        estimators=[(name, model)],
        parameter_space=parameter_space, 
        optimize=True,
        R=10, 
        n_outer=5, 
        n_inner=3,
        seed=42
    )
    
    single_run.runcv()
    single_run.results[name].to_csv(f"../data/Task3/tunedraw_{name}.csv", index=False) #save the 50 values of each metric in a csv file
    temp_dict = {name: single_run.results[name]}
    
    #Save the median and the CI intervals for each metric
    get_models_statistics(temp_dict, output_filename = f"../data/Task3/tunedboot_results_{name}.csv")
    print(f"Finished {name}. Data saved\n")

In [ ]:
#I will combine all the csv files with the metrics values for each model into one file 
raw_files = glob.glob("../data/Task3/tunedraw_*.csv")

#Concatenate these raw files
all_raws = [pd.read_csv(f) for f in raw_files]
combined_raw_df = pd.concat(all_raws).reset_index(drop=True)

#Save the final file
combined_raw_df.to_csv("../data/Task3/final_tuned_raw_comparison_table.csv", index=False)


#Combine all the files with the models statistics (after applying bootstraping)
summary_files = glob.glob("../data/Task3/tunedboot_*.csv") 

#Concatenate these files into one DataFrame
all_summaries = [pd.read_csv(f) for f in summary_files]
combined_stats_df = pd.concat(all_summaries).reset_index(drop=True)

#Save
combined_stats_df.to_csv("../data/Task3/final_tuned_comparison_table.csv", index=False)

#Plot the results
plot_metrics(combined_stats_df, filename="../figures/Task3/final_tuned_comparison.png")